In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import wittgenstein as lw

# --- 1. Tải tập dữ liệu Iris ---
iris = load_iris()
df = pd.DataFrame(iris.data, columns=iris.feature_names)
df['target'] = iris.target
df['target'] = df['target'].map({0: 'setosa', 1: 'versicolor', 2: 'virginica'})

X = df.drop('target', axis=1)
y = df['target']

# --- 2. Chia tập dữ liệu ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# --- 3. Huấn luyện thuật toán RIPPER (Tự làm One-vs-Rest) ---
models = {}
classes = ['setosa', 'versicolor', 'virginica']

print("--- CÁC LUẬT PHÂN LOẠI (LEARNED RULES) ---")
for cls in classes:
    # Bước quan trọng: Tạo nhãn nhị phân. 
    # Nếu là class đang xét thì đánh True (1), ngược lại là False (0)
    y_train_binary = (y_train == cls)
    
    # Huấn luyện mô hình cho riêng class này
    clf = lw.RIPPER(random_state=42)
    clf.fit(X_train, y_train_binary)
    models[cls] = clf
    
    # In ra luật
    print(f"\nLuật để nhận diện [{cls}]:")
    clf.out_model()
    
print("-" * 50)

# --- 4 & 5. Dự đoán & Đánh giá ---
# Tạo một DataFrame để chứa xác suất dự đoán của từng class
probs = pd.DataFrame(index=X_test.index)

# Lấy xác suất dự đoán (predict_proba) từ từng mô hình
for cls, clf in models.items():
    # Cột 1 chứa xác suất của nhãn True (tức là xác suất rơi vào class đang xét)
    probs[cls] = clf.predict_proba(X_test)[:, 1]

# Lựa chọn class có xác suất cao nhất cho mỗi dòng
y_pred = probs.idxmax(axis=1)

print("\n--- ĐÁNH GIÁ MÔ HÌNH ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

--- CÁC LUẬT PHÂN LOẠI (LEARNED RULES) ---

Luật để nhận diện [setosa]:
[[petalwidth(cm)=<0.2] V
[petalwidth(cm)=0.2-0.4] V
[petallength(cm)=1.5-1.67]]

Luật để nhận diện [versicolor]:
[[petallength(cm)=4.25-4.6] V
[petallength(cm)=3.96-4.25] V
[petallength(cm)=1.67-3.96 ^ petalwidth(cm)=0.4-1.2] V
[petallength(cm)=4.6-5.0 ^ petalwidth(cm)=1.3-1.5]]

Luật để nhận diện [virginica]:
[[petallength(cm)=>5.81] V
[petallength(cm)=5.32-5.81] V
[petallength(cm)=5.0-5.32] V
[petallength(cm)=4.6-5.0 ^ sepalwidth(cm)=<2.5] V
[petalwidth(cm)=1.5-1.8 ^ sepallength(cm)=6.0-6.3]]
--------------------------------------------------

--- ĐÁNH GIÁ MÔ HÌNH ---
Accuracy: 0.9333

Confusion Matrix:
[[10  0  0]
 [ 0  9  1]
 [ 0  1  9]]

Classification Report:
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       0.90      0.90      0.90        10
   virginica       0.90      0.90      0.90        10

    accuracy                        

Luật nào dễ diễn giải nhất? (Which rule is the easiest to interpret?)

Trả lời: Luật để nhận diện lớp setosa là dễ diễn giải nhất. Nhìn vào kết quả, luật của setosa bao gồm các điều kiện rất đơn giản và ngắn gọn, chủ yếu xoay quanh việc đài hoa/cánh hoa có kích thước nhỏ (ví dụ: petalwidth <= 0.2 hoặc 0.2 - 0.4). Các điều kiện này chỉ dùng toán tử HOẶC (V), không bị lồng ghép phức tạp bởi các toán tử VÀ (^) như quy tắc của versicolor hay virginica. Điều này giúp con người đọc vào là hiểu ngay lập tức ranh giới để phân loại hoa setosa.

Lớp Iris nào được phân loại tốt nhất? (Which Iris class is classified best?)

Trả lời: Lớp setosa luôn là lớp được phân loại tốt nhất (nếu xem kỹ trên Classification Report đầy đủ, F1-score của lớp này thường đạt mức tuyệt đối 1.0). Lý do là vì xét trên không gian đặc trưng, dữ liệu của hoa Setosa tách biệt hoàn toàn thành một cụm riêng so với hai loại Versicolor và Virginica (hai loại này thường có ranh giới chồng lấn lên nhau một chút).

Ưu điểm chính của RIPPER so với các mô hình hộp đen? (What is the main advantage of RIPPER compared with black-box models?)

Trả lời: Ưu điểm vượt trội nhất là khả năng diễn giải tuyệt đối (White-box model). Các mô hình hộp đen như Mạng nơ-ron (Neural Networks) có thể cho kết quả đúng nhưng không thể giải thích tại sao chúng lại dự đoán như vậy. Ngược lại, RIPPER trích xuất ra các luật logic dạng văn bản (Nếu A và B thì C). Nhờ đó, người dùng có thể dễ dàng kiểm chứng tính hợp lý của mô hình, đồng thời dễ dàng chuyển đổi các luật này thành các câu lệnh IF-THEN trong lập trình (như C++, Python) hoặc câu truy vấn SQL để tích hợp vào hệ thống thực tế.

In [5]:
# Danh sách các tỷ lệ test_size theo yêu cầu (70:30, 75:25, 80:20)
test_sizes = [0.30, 0.25, 0.20]
classes = ['setosa', 'versicolor', 'virginica']

print("--- SO SÁNH SỐ LƯỢNG LUẬT VỚI CÁC TỶ LỆ CHIA KHÁC NHAU ---")

for size in test_sizes:
    # 1. Chia lại dữ liệu
    X_train_ext, X_test_ext, y_train_ext, y_test_ext = train_test_split(
        X, y, test_size=size, random_state=42, stratify=y
    )
    
    total_rules = 0
    
    # 2. Huấn luyện One-vs-Rest giống hệt phần trên
    for cls in classes:
        # Tạo nhãn nhị phân: True nếu đúng class đang xét, False nếu là class khác
        y_train_binary = (y_train_ext == cls)
        
        # Khởi tạo và fit mô hình cho 1 class
        clf_ext = lw.RIPPER(random_state=42)
        clf_ext.fit(X_train_ext, y_train_binary)
        
        # Cộng dồn số luật được tạo ra cho class này
        total_rules += len(clf_ext.ruleset_)
        
    # Tính tỷ lệ phần trăm để in ra cho đẹp
    train_ratio = int((1 - size) * 100)
    test_ratio = int(size * 100)
    
    print(f"Tỷ lệ Train:Test = {train_ratio}:{test_ratio} | Tổng số luật sinh ra (cho cả 3 loại): {total_rules}")

--- SO SÁNH SỐ LƯỢNG LUẬT VỚI CÁC TỶ LỆ CHIA KHÁC NHAU ---
Tỷ lệ Train:Test = 70:30 | Tổng số luật sinh ra (cho cả 3 loại): 14
Tỷ lệ Train:Test = 75:25 | Tổng số luật sinh ra (cho cả 3 loại): 10
Tỷ lệ Train:Test = 80:20 | Tổng số luật sinh ra (cho cả 3 loại): 12


Tổng kết Mô hình phân loại RIPPER:

Hiệu suất dự đoán (Predictive performance): Mô hình hoạt động khá hiệu quả với tập dữ liệu nhỏ và cấu trúc đơn giản như Iris, đạt độ chính xác ~93.3%. Dù vậy, hiệu suất của nó có thể hơi kém cạnh một chút nếu đem so sánh với các thuật toán học máy phức tạp hơn (như Random Forest hay SVM) trên các tập dữ liệu có độ nhiễu cao.

Khả năng diễn giải (Interpretability): Xuất sắc. Khả năng xuất ra các quy tắc phân loại rõ ràng (dưới dạng Disjunctive Normal Form - DNF) giúp các nhà phân tích dữ liệu và ngay cả người không có chuyên môn kỹ thuật cũng có thể hiểu được cơ sở ra quyết định của mô hình.

Điểm mạnh (Strengths):

Rất minh bạch, logic rõ ràng.

Khả năng loại bỏ các đặc trưng không quan trọng (feature selection ngầm) khi xây dựng luật.

Dễ dàng xuất và nhúng các luật (rules) trực tiếp vào các phần mềm quản lý/vận hành doanh nghiệp.

Hạn chế (Limitations):

Thuật toán mặc định gặp khó khăn với phân loại đa lớp (Multi-class classification) và thường yêu cầu phải chia nhỏ bài toán bằng phương pháp One-vs-Rest.

Với dữ liệu số liên tục (continuous data) như tập Iris, thuật toán phải tự động chia khoảng (binning), đôi khi dẫn đến việc ranh giới phân loại bị cứng nhắc.

Dễ bị "bùng nổ" số lượng luật (sinh ra quá nhiều điều kiện nhỏ lẻ) nếu dữ liệu quá phức tạp và không được cắt tỉa (pruning) hợp lý.